In [1]:
from backend.mcp_servers.indexing.parsers.pdf import parse_pdf

In [2]:
pdf = parse_pdf("Profile.pdf")

In [ ]:
import asyncio
import os
import uuid

from dotenv import load_dotenv

from a2a.client import ClientConfig, ClientFactory

from a2a.types import (
    Message,
    Part,
    SendMessageRequest,
)


load_dotenv()


A2A_AGENT_URL = os.getenv(
    "A2A_AGENT_URL",
    "http://localhost:9999",
)

DOCUMENT_ID = "061d7f0a-a11e-4207-94cd-6c2b06cacfee"
#os.getenv(
    # "DOCUMENT_ID",
#)


def create_user_message(text: str) -> Message:
    """
    Create a valid A2A user message.
    """

    return Message(
        message_id=str(uuid.uuid4()),
        role=Role.ROLE_USER,
        parts=[
            Part(
                text=text
            )
        ],
    )


async def main():

    if not DOCUMENT_ID:
        raise RuntimeError(
            "DOCUMENT_ID environment variable is required."
        )

    print(
        f"Connecting to A2A agent: "
        f"{A2A_AGENT_URL}"
    )

    # ---------------------------------------------------------
    # Create A2A client
    # ---------------------------------------------------------

    config = ClientConfig(
        streaming=False,
    )

    factory = ClientFactory(
        config=config,
    )

    client = await factory.create_from_url(
        A2A_AGENT_URL,
    )

    print(
        "Connected to A2A agent."
    )

    # ---------------------------------------------------------
    # Ask question
    # ---------------------------------------------------------

    question = input(
        "\nQuestion: "
    ).strip()

    if not question:
        print(
            "No question provided."
        )
        await client.close()
        return

    # ---------------------------------------------------------
    # Create A2A message
    # ---------------------------------------------------------

    message = create_user_message(
        question
    )

    print(
        f"\nMessage ID: {message.message_id}"
    )

    # ---------------------------------------------------------
    # Create request
    # ---------------------------------------------------------

    request = SendMessageRequest(
        message=message,
        metadata={
            "document_id": DOCUMENT_ID,
        },
    )

    # ---------------------------------------------------------
    # Send
    # ---------------------------------------------------------

    print(
        "\nSending request...\n"
    )

    try:

        async for response in client.send_message(
            request
        ):

            print(
                "A2A RESPONSE:"
            )

            print(
                response
            )

    except Exception as exc:

        print(
            "\nA2A request failed:"
        )

        print(
            type(exc).__name__
        )

        print(
            exc
        )

        raise

    finally:

        await client.close()


await main()

In [5]:
from backend.apps.retrieval_agent.agent import RetrievalAgent
DOCUMENT_ID = "c1e468af-c760-4d61-bfa5-eed7ed72720f"
question = "Return all available essential information into a markdown format. " #"Return the top best models used and their scores. Also what do the authors achieve?"

async def main():

    agent = RetrievalAgent()

    answer = await agent.answer_query(
        document_id=DOCUMENT_ID,
        question=question,
    )

    print("\n==============================")
    print("FINAL ANSWER")
    print("==============================")
    return answer

answer = await main()
answer["answer"]

Query repeated. Stopping retrieval loop: What is the name of the university where Nick Giannakis received his PhD in Artificial Intelligence and Machine Learning in Economics?



FINAL EVIDENCE

    SOURCE 1

    TEXT:
    Contact
Nikolaos.Giannakis@mail.com
www.linkedin.com/in/nick-
giannakis-cv (LinkedIn)
Top Skills
TypeScript
C#
JavaScript
Languages
Greek (Native or Bilingual)
English (Native or Bilingual)
German
Certifications
JALLC Lessons Learned
Machine Learning Scientist with
Python Track
TensorFlow in Practice Professional
Certificate
Microsoft Certified: Power Bi Data
Analyst Associate
Natural Language Processing
Specialization
Publications
Neuropsychiatric symptoms
influence performance of activities
of daily living in symptomatic
Alzheimer’s Disease
Nick Giannakis
PhD Student AI in Economics
Athens Metropolitan Area
Summary
Data Science enthusiast familiar with gathering, cleaning, organizing
and modelling data. Advanced understanding of statistical, algebraic
and other analytical techniques. Highly organized, motivated
and diligent with significant background in Artificial Intelligence
techniques.

    SCORE:
    0.5105668495657099

    METADATA:


"### Essential Information in Markdown Format\n\n#### Contact Information\n| Category | Value |\n| --- | --- |\n| Email | Nikolaos.Giannakis@mail.com |\n| LinkedIn Profile | www.linkedin.com/in/nick-giannakis-cv |\n\n#### Education\n| Institution | Degree | Field of Study |\n| --- | --- | --- |\n| Democritus University of Thrace - (D.U.Th.) | PhD | Artificial Intelligence and Machine Learning in Economics |\n| University of Derby | MSc | Big Data Analytics |\n| Aristotle University of Thessaloniki (AUTH) | MSc | Artificial Intelligence |\n| MBA University of the Aegean | MBA | Operations Management |\n\n#### Work Experience\n| Company | Position | Dates |\n| --- | --- | --- |\n| Stout | Vice President, Full Stack Data Analyst, Valuation Advisory | October 2025 - Present |\n| bAnt | Data Analyst | September 2021 - October 2024 |\n| Hellenic Army | Transportation Operations Officer | August 2020 - January 2023 |\n| NATO | CBRN Staff Officer | July 2017 - August 2020 |\n\n#### Skills\n| S

In [3]:
import pandas as pd
print(answer["retrieval_iterations"])
pd.DataFrame(answer["sources"])

2


,text,score,metadata
0,Contact\nNikolaos.Giannakis@mail.com\nwww.link...,0.500167,{'document_id': 'c1e468af-c760-4d61-bfa5-eed7e...
1,Contact\nNikolaos.Giannakis@mail.com\nwww.link...,0.484713,{'document_id': 'c1e468af-c760-4d61-bfa5-eed7e...
2,"Advanced understanding of statistical, algebra...",0.479374,{'document_id': 'c1e468af-c760-4d61-bfa5-eed7e...
3,Experience\nStout\n3 years 10 months\nVice Pre...,0.475152,{'document_id': 'c1e468af-c760-4d61-bfa5-eed7e...
4,Education\nDemocritus University of Thrace - (...,0.443501,{'document_id': 'c1e468af-c760-4d61-bfa5-eed7e...


In [4]:
answer["answer"]

'Here is the available essential information in JSON format:\n\n```\n{\n  "name": "Nikolaos Giannakis",\n  "email": "Nikolaos.Giannakis@mail.com",\n  "linkedin": "www.linkedin.com/in/nick-giannakis-cv",\n  "skills": ["TypeScript", "C#", "JavaScript"],\n  "languages": ["Greek (Native or Bilingual)", "English (Native or Bilingual)", "German"],\n  "certifications": [\n    "JALLC Lessons Learned",\n    "Machine Learning Scientist with Python Track",\n    "TensorFlow in Practice Professional Certificate",\n    "Microsoft Certified: Power Bi Data Analyst Associate"\n  ],\n  "experience": [\n    {\n      "company": "Stout",\n      "position": "Vice President, Full Stack Data Analyst, Valuation Advisory",\n      "dates": "October 2025 - Present (8 months)"\n    },\n    {\n      "company": "Stout",\n      "position": "Associate, Full Stack Data Analyst, Valuation Advisory",\n      "dates": "August 2022 - October 2025 (3 years 3 months)"\n    },\n    {\n      "company": "bAnt",\n      "position"

In [ ]:
from backend.apps.client.agent import MCPAgent
from backend.apps.client.config import AgentConfig

# Load config (reads servers.json + .env)
config = AgentConfig.load()

async with MCPAgent(config) as agent:
    # Ask a question; agent decides which tools to use
    answer = await agent.run("Upload and index this PDF document")